# Orchestrating a Fraud Detection Model Lifecycle with cuDF, Prefect, MLflow, and Triton

_April, 2026_

A production MLOps pipeline for the [NVIDIA Financial Fraud Detection AI Blueprint](https://github.com/NVIDIA-AI-Blueprints/financial-fraud-detection), built with [Prefect](https://www.prefect.io/), [MLflow](https://mlflow.org/), and [Triton Inference Server](https://github.com/triton-inference-server/server).

The blueprint provides a [notebook](https://github.com/NVIDIA-AI-Blueprints/financial-fraud-detection/blob/main/notebooks/financial-fraud-usage-v2.ipynb) that manually walks through data preprocessing, GNN + XGBoost model training, Triton serving, and inference evaluation. This example takes that workflow and wraps each stage in automated orchestration with experiment tracking and continuous deployment: the kind of infrastructure you'd build around it for production use.

## What the Blueprint Does

The blueprint's notebook runs through these steps manually:

1. **Load and preprocess** 24M credit card transactions from the [IBM TabFormer](https://github.com/IBM/TabFormer) dataset using cuDF, building a bipartite User ↔ Merchant graph where edges represent transactions
2. **Train** a 2-hop SAGEConv GNN that produces node embeddings, then an XGBoost classifier on those embeddings for edge classification ("is this transaction fraudulent?")
3. **Serve** the trained model on Triton Inference Server with a custom Python backend
4. **Evaluate** by sending held-out test data to Triton and computing F1, precision, recall, and accuracy

Each step is a notebook cell that you run manually and inspect the output. There's no orchestration, no experiment tracking, no way to compare model versions, and no automated deployment.

This example automates the entire flow and adds the production concerns:

- **Orchestration**: [Prefect](https://docs.prefect.io) flows chain the stages together, track runs, and handle failures
- **Experiment tracking**: [MLflow](https://mlflow.org/docs/latest) logs hyperparameters, model artifacts, and evaluation metrics for every run
- **Continuous deployment**: champion/challenger evaluation via Triton's native model versioning, with automatic promotion or rollback

## Architecture

The pipeline has four stages, each implemented as a Prefect [flow](https://docs.prefect.io/3.0/develop/write-flows) composed of reusable [tasks](https://docs.prefect.io/3.0/develop/write-tasks):

```text
full_pipeline_flow
├── preprocess_flow    →  cuDF graph formation + MLflow metadata logging
├── train_flow         →  NGC training container + MLflow experiment logging
├── evaluate_flow      →  Triton champion/challenger scoring + MLflow metrics
└── deploy_flow        →  Promote or rollback + MLflow model registry
```

The full pipeline runs all four as subflows in sequence, passing results between them. Each stage can also run independently for ad-hoc experiments.

### Infrastructure

Three services support the pipeline, all running as Docker containers via `docker-compose.yml`:

| Service | Role | Port |
|---------|------|------|
| **Prefect** | Flow orchestration, run tracking, scheduling, UI | 4200 |
| **MLflow** | Experiment tracking, artifact storage, model registry | 5050 |
| **Triton** | GPU inference, model versioning, model control API | 8000/8001/8002 |

### Champion/Challenger via Triton Native Versioning

The evaluate stage uses Triton's built-in model versioning for zero-downtime model comparison. Triton's model repository supports multiple version directories (`1/`, `2/`, `3/`...), and with `version_policy: { all: {} }` in `config.pbtxt`, all versions are served simultaneously.

The flow works like this:

1. Copy newly trained model artifacts as version N+1 in the model repository
2. Reload the model in Triton so that both version N (champion) and N+1 (challenger) are live
3. Send the same held-out test data to both versions using version-specific inference requests
4. Compare F1 scores and decide whether the challenger should be promoted
5. Remove the loser's version directory and reload Triton

No separate evaluation server is needed. Triton serves both models simultaneously during comparison.

## Project Structure

```text
fraud-detection-mlops/
├── pipeline/
│   ├── config.py              Paths, service URLs, hyperparameters, thresholds
│   ├── deployments.py         Register flows with work pools and schedules
│   ├── flows/
│   │   ├── preprocess.py      Stage 1: cuDF graph formation + MLflow
│   │   ├── train.py           Stage 2: NGC container + MLflow logging
│   │   ├── evaluate.py        Stage 3: Triton versioning + champion/challenger scoring
│   │   ├── deploy.py          Stage 4: promote or rollback
│   │   └── full_pipeline.py   Orchestrator: chains all stages as subflows
│   └── tasks/
│       ├── data.py            Test data loading for evaluation
│       ├── training.py        Config generation + Docker container lifecycle
│       ├── mlflow_utils.py    Experiment tracking + model registry
│       └── triton.py          Version staging, model reload, inference scoring
├── scripts/
│   ├── preprocess_tabformer.py    cuDF preprocessing (adapted from blueprint)
│   └── download_data.sh           TabFormer dataset download helper
├── triton/
│   └── Dockerfile                 Custom Triton image with PyTorch, XGBoost GPU, PyG
├── docker-compose.yml             MLflow + Prefect + Triton services
├── environment.yml                Conda environment (RAPIDS + pipeline deps)
└── .env.example                   Configuration template
```

The `tasks/` layer contains reusable units of work (data loading, container execution, MLflow logging, Triton management). The `flows/` layer composes tasks into pipeline stages. This separation is important for scaling: tasks can be distributed across workers and [work pools](https://docs.prefect.io/3.0/deploy/infrastructure-concepts/work-pools), while flows define the orchestration logic.

## Prerequisites

- **NVIDIA GPU**: Turing architecture or newer (T4, A10G, L4, A100, H100). Compute capability >= 7.5 is required by the training container.
- **Docker** with [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html)
- **NGC API key** from [ngc.nvidia.com/setup/api-key](https://ngc.nvidia.com/setup/api-key) (needed to pull the training container)
- **Conda**: [miniforge](https://github.com/conda-forge/miniforge) recommended


## Setup

### Clone and configure

```console
$ git clone https://github.com/rapidsai/deployment.git
$ cd fraud-detection-mlops
$ cp .env.example .env
```

Edit `.env` to set your `NGC_API_KEY` and adjust data paths for your environment.

### Install Python dependencies

All dependencies (RAPIDS, Prefect, MLflow, Triton client) are specified in `environment.yml`:

```console
$ conda env create -f environment.yml
$ conda activate fraud-mlops
```


### Download the TabFormer dataset

The dataset is 266MB of synthetic credit card transactions (24M rows, 15 columns) from [IBM TabFormer](https://github.com/IBM/TabFormer):

```console
$ ./scripts/download_data.sh ./data/TabFormer
```

This prompts you to download `transactions.tgz` from [IBM Box](https://ibm.ent.box.com/v/tabformer-data/folder/130747715605) and extract it.

### Start the infrastructure

```console
# MLflow + Prefect (no GPU needed)
$ docker compose up -d

# With Triton (needs GPU and a pre-built image, see below)
$ docker compose --profile gpu up -d
```

Verify the services are running:
- Prefect UI: [http://localhost:4200](http://localhost:4200)
- MLflow UI: [http://localhost:5050](http://localhost:5050)

### Pull the training container

```console
$ echo "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin
$ docker pull nvcr.io/nvidia/cugraph/financial-fraud-training:2.0.0
```

### Build the Triton serving image

The custom Triton image (`triton/Dockerfile`) installs PyTorch 2.7, XGBoost 3.0 (GPU), PyTorch Geometric 2.6, and Captum on top of the Triton 25.04 base image:

```console
$ docker build -t triton-fraud:latest triton/
```

This build takes 15-20 minutes and produces a ~24GB image.

## Stage 1: Preprocessing

**Flow:** `pipeline/flows/preprocess.py` | **Script:** `scripts/preprocess_tabformer.py`

The preprocessing stage takes the raw TabFormer CSV and produces graph-structured data for the GNN training container.

The script (`scripts/preprocess_tabformer.py`, adapted from the blueprint's `preprocess_TabFormer_lp.py`) uses **cuDF** for GPU-accelerated DataFrame operations on the 24M-row dataset:

1. **Clean column names**: strip spaces, standardize field names
2. **Encode categorical features**: one-hot encoding for low-cardinality fields (< 8 categories), binary encoding for high-cardinality fields
3. **Build the bipartite graph**: Users and Merchants become nodes, each transaction becomes an edge with encoded attributes and a fraud/not-fraud label
4. **Undersample**: balance the dataset to a configurable fraud ratio (default 10%)
5. **Split by year**: pre-2018 for training, 2018 for validation, post-2018 for testing
6. **Write CSVs**: node features, edge indices, edge attributes, edge labels, and feature masks to `gnn/` and `gnn/test_gnn/`

The Prefect flow wraps this script and logs preprocessing metadata (row counts, graph dimensions) to MLflow:

```python
# pipeline/flows/preprocess.py
@flow(name="preprocess", log_prints=True)
def preprocess_flow(
    raw_csv_path: str = RAW_CSV_PATH,
    output_base_path: str = DATA_ROOT,
    fraud_ratio: float = 0.1,
    under_sample: bool = True,
) -> dict:
    metadata, user_mask_map, mx_mask_map, tx_mask_map = preprocess_data(...)
    # Log to MLflow
    experiment_id = get_or_create_experiment()
    with mlflow.start_run(experiment_id=experiment_id, run_name="preprocess"):
        mlflow.log_params({"preprocess.fraud_ratio": fraud_ratio, ...})
        mlflow.log_metrics({"preprocess.num_transactions": float(metadata["num_transactions"]), ...})
```

The cuDF import is deferred. It only happens when the preprocessing flow actually runs, so all other pipeline stages work fine on CPU-only machines.

## Stage 2: Training

**Flow:** `pipeline/flows/train.py` | **Tasks:** `pipeline/tasks/training.py`, `pipeline/tasks/mlflow_utils.py`

Training uses NVIDIA's `financial-fraud-training:2.0.0` NGC container as a black box. The pipeline generates the config it expects, runs the container, and collects the artifacts.

### Config generation

The `generate_training_config` task builds a JSON config from the flow's hyperparameters:

```python
# Default hyperparameters from pipeline/config.py
DEFAULT_GNN_PARAMS = {
    "hidden_channels": 32, "n_hops": 2, "layer": "SAGEConv",
    "dropout_prob": 0.1, "batch_size": 4096, "fan_out": 10, "num_epochs": 8,
}
DEFAULT_XGB_PARAMS = {
    "max_depth": 6, "learning_rate": 0.2, "num_parallel_tree": 3,
    "num_boost_round": 512, "gamma": 0.0,
}
```

### Container execution

The `run_training_container` task runs the NGC container with volume mounts for graph data, output directory, and config:

```python
# pipeline/tasks/training.py
cmd = [
    "docker", "run", "--rm", "--gpus", "device=0",
    "--cap-add", "SYS_NICE", "--shm-size=8g", "--privileged",
    "-e", "NCCL_IB_DISABLE=1", "-e", "NCCL_NET=Socket",  # Cloud NCCL workarounds
    "-v", f"{data_dir}:/data",
    "-v", f"{output_dir}:/trained_models",
    "-v", f"{config_path}:/app/config.json",
    "--entrypoint", "bash", training_image,
    "-c", "torchrun --standalone --nproc_per_node=1 /app/main.py --config /app/config.json",
]
```

The NCCL environment variables (`NCCL_IB_DISABLE`, `NCCL_NET=Socket`, etc.) work around a crash that occurs on cloud instances without InfiniBand hardware. The container's bundled UCX library expects HPC networking that isn't present on standard AWS/GCP VMs.

### MLflow logging

After training completes, the flow logs everything to MLflow:

- **Hyperparameters**: `gnn.hidden_channels`, `gnn.n_hops`, `xgb.max_depth`, `xgb.num_boost_round`, etc.
- **Artifacts**: the training config JSON and the full model directory
- **Tags**: `trigger: automated` (or custom tags for ad-hoc experiments)

The flow returns the MLflow run ID, which the evaluate stage uses to associate metrics with the same run.

## Stage 3: Evaluation

**Flow:** `pipeline/flows/evaluate.py` | **Tasks:** `pipeline/tasks/triton.py`, `pipeline/tasks/data.py`

The evaluation stage compares the newly trained model (challenger) against the currently deployed model (champion) using Triton's native versioning.

### Staging the challenger

The `stage_challenger_version` task copies the new model's artifacts as the next version directory in Triton's model repository. If the champion is version 1, the challenger becomes version 2:

```text
prediction_and_shapley/
├── 1/          ← champion (currently serving)
│   ├── model.py
│   ├── state_dict_gnn_model.pth
│   └── embedding_based_xgboost.json
├── 2/          ← challenger (just staged)
│   ├── model.py
│   ├── state_dict_gnn_model.pth
│   └── embedding_based_xgboost.json
└── config.pbtxt  (version_policy { all {} })
```

The task also ensures `config.pbtxt` has `version_policy { all {} }` so Triton serves both versions simultaneously. Without this, Triton defaults to `latest: { num_versions: 1 }` and would drop the champion when the challenger is loaded.

### Scoring

After reloading the model in Triton, the flow loads the held-out test data (25,803 samples, ~8% fraud) and sends it to both versions using Triton's version-specific inference:

```python
# pipeline/tasks/triton.py
response = client.infer(
    model_name,
    inputs=inputs,
    model_version=str(version),  # "1" for champion, "2" for challenger
    outputs=outputs,
)
```

Each version returns fraud probability scores. The task applies a decision threshold (default 0.5) and computes F1, precision, recall, and accuracy against the ground truth labels.

### Promotion decision

The flow compares the challenger's F1 score against the champion's. Promotion requires the challenger to exceed the champion by at least `min_improvement` (default 0.0):

```python
should_promote = challenger_f1 > champion_f1 + min_improvement
```

Both sets of metrics, the deltas, and the promotion decision are logged to the same MLflow run that was created during training.

## Stage 4: Deployment

**Flow:** `pipeline/flows/deploy.py`

The deploy stage acts on the evaluation result.

### If the challenger wins (promote)

1. Remove the old champion's version directory from the model repository
2. Reload the model in Triton (only the challenger version remains)
3. Run a health check to verify the new champion is responsive
4. Register the model version in MLflow's model registry and assign the `champion` alias

If the health check fails, the flow raises an error. This surfaces in Prefect as a failed run that you can investigate.

### If the champion wins (reject)

1. Remove the challenger's version directory
2. Reload Triton to restore the champion-only state

In both cases, Triton ends up serving exactly one version: the winner. The MLflow model registry provides an authoritative record of which model version is in production via the `champion` alias:

```python
# pipeline/tasks/mlflow_utils.py
mv = mlflow.register_model(f"runs:/{run_id}/model", model_name)
client.set_registered_model_alias(model_name, "champion", mv.version)
```

## Running the Pipeline

### Full pipeline

Run all four stages end-to-end:

```console
$ python -m pipeline.flows.full_pipeline
```

This chains preprocess → train → evaluate → deploy as subflows, passing results between stages automatically. If graph data already exists from a previous preprocessing run, set `skip_preprocess=True` in the flow parameters.

### Individual stages

Each stage can be run independently for debugging or ad-hoc experiments:

```console
# Preprocess raw CSV into graph data
$ python -m pipeline.flows.preprocess

# Train with default hyperparameters
$ python -m pipeline.flows.train

# Evaluate a specific training run
$ python -m pipeline.flows.evaluate <mlflow_run_id>

# Deploy based on evaluation result
$ python -m pipeline.flows.deploy '{"should_promote": true, ...}'
```

### Experimenting with hyperparameters

To try different model configurations, pass custom hyperparameters to the train flow:

```python
from pipeline.flows.train import train_flow

run_id = train_flow(
    gnn_params={"hidden_channels": 64, "n_hops": 3, "num_epochs": 12, ...},
    xgb_params={"max_depth": 8, "num_boost_round": 1024, ...},
)
```

Each run is tracked in MLflow with full hyperparameters and metrics, making it straightforward to compare experiments. The evaluate stage will then score this new model against the current champion.

## Monitoring and Iteration

### Prefect UI

The [Prefect UI](http://localhost:4200) at port 4200 tracks every flow run. Here is a completed full pipeline run showing the three subflows (train, evaluate, deploy) with their execution timeline:

![Prefect flow run showing subflow hierarchy](images/prefect-flow-runs.png)

The UI also shows task-level logs from each step, which is useful for debugging container failures or Triton connection issues.

### MLflow UI

The [MLflow UI](http://localhost:5050) at port 5050 tracks experiments across runs. The experiment table shows all training runs with their hyperparameters and evaluation metrics side by side:

![MLflow experiment table with training runs](images/mlflow-experiment-table.png)

Clicking into a run shows the full details: all GNN and XGBoost hyperparameters, challenger and champion metrics, deltas, and the promotion decision:

![MLflow run details with parameters and metrics](images/mlflow-run-details.png)

The model registry tracks which version is in production. The `champion` alias always points to the current production model:

![MLflow model registry with champion alias](images/mlflow-model-registry.png)

### Iteration workflow

A typical iteration cycle:

1. Run the full pipeline with default hyperparameters
2. Check MLflow for the baseline F1 score
3. Modify hyperparameters (deeper GNN, more XGBoost rounds, different learning rate)
4. Run train + evaluate to see if the new model beats the champion
5. If promoted, the new model is automatically serving in Triton

## Scaling to Multiple Machines

The default mode runs all stages as subflows in a single process on one GPU machine. For multi-machine setups, `pipeline/deployments.py` registers each flow as a Prefect [deployment](https://docs.prefect.io/3.0/deploy/infrastructure-concepts/deployments) with its own work pool and schedule.

### How it works

The evaluate and deploy stages need both filesystem access to Triton's model repository (for staging model versions) and HTTP access to the Triton API (for reloading models and running inference). Because of this, all pipeline stages must run on the same machine as Triton.

The recommended multi-machine setup runs the Prefect and MLflow servers on your local machine (e.g., your laptop) so you get browser access to both UIs without port forwarding. The GPU instance runs all pipeline flows and Triton. The local machine is purely for monitoring and triggering runs.

- **Local machine**: Prefect server, MLflow server, browser access to both UIs
- **GPU instance**: all pipeline flows (gpu pool worker), Triton server

### Setting up deployments

**1. Start the services on the local machine:**

```console
$ docker compose up -d
```

This starts Prefect (port 4200) and MLflow (port 5050). Both UIs are available at `localhost` in your browser.

**2. Point the GPU instance at the local machine** by setting the API URLs:

```console
# On the GPU instance (replace with the local machine IP)
$ export PREFECT_API_URL=http://LOCAL_MACHINE_IP:4200/api
$ export MLFLOW_TRACKING_URI=http://LOCAL_MACHINE_IP:5050
```

**3. Create the work pool** (run once, from the local machine):

```console
$ prefect work-pool create gpu --type process
```

**4. Register and serve all deployments:**

```console
$ python -m pipeline.deployments
```

This registers 5 deployments:
- `preprocess` (gpu pool, nightly at midnight)
- `train` (gpu pool, on-demand)
- `evaluate` (gpu pool, on-demand)
- `deploy` (gpu pool, on-demand)
- `full-pipeline` (gpu pool, nightly at 2am)

**5. Start a worker on the GPU instance:**

```console
$ prefect worker start --pool gpu
```

You can trigger runs from the Prefect UI on the local machine or via the CLI:

```console
$ prefect deployment run full-pipeline/full-pipeline
```

The pipeline code does not change between single-machine and multi-machine modes. The difference is that flow runs are tracked and triggered via the Prefect server on the local machine, while all execution happens on the GPU instance.

## Conclusion

This example demonstrates how to take an ML notebook and build production infrastructure around it:

- **Prefect** orchestrates the pipeline stages, tracks runs, and enables scheduling
- **MLflow** tracks every experiment (hyperparameters, artifacts, metrics) and manages the model registry
- **Triton** serves models with native versioning, enabling champion/challenger evaluation without separate infrastructure

The same patterns (flow orchestration, experiment tracking, versioned deployment) apply to any ML pipeline where you need to automate training, compare model versions, and deploy the winner.

### Resources

- [NVIDIA Financial Fraud Detection AI Blueprint](https://github.com/NVIDIA-AI-Blueprints/financial-fraud-detection)
- [IBM TabFormer dataset](https://github.com/IBM/TabFormer)
- [Prefect 3.x documentation](https://docs.prefect.io)
- [MLflow documentation](https://mlflow.org/docs/latest)
- [Triton Inference Server](https://github.com/triton-inference-server/server)